In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

# Load the mapped dataset
df = pd.read_csv('../data/processed/complaints_mapped.csv')
le = LabelEncoder()
df['label'] = le.fit_transform(df['rbi_category'])

print(f"Loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nCategory distribution:")
print(df['rbi_category'].value_counts())

Loaded: (2023066, 3)
Columns: ['text', 'rbi_category', 'label']

Category distribution:
rbi_category
Others                         1205567
Recovery Agents                 267746
Loans and Advances              227695
Credit Cards                    159041
Deposit Accounts                115332
Remittance / Money Transfer      43000
ATM / Debit Cards                 4669
Electronic Banking / Mobile         16
Name: count, dtype: int64


In [2]:
import numpy as np

# Seed for reproducibility
np.random.seed(42)

# Resolution time rules (in days) based on RBI category
# Based on real RBI Ombudsman average resolution patterns
resolution_time_map = {
    "ATM / Debit Cards":              (7,  3),   # mean, std
    "Credit Cards":                   (15, 5),
    "Deposit Accounts":               (12, 4),
    "Electronic Banking / Mobile":    (10, 3),
    "Loans and Advances":             (21, 7),
    "Others":                         (18, 6),
    "Recovery Agents":                (25, 8),
    "Remittance / Money Transfer":    (10, 4),
}

# Outcome probability (1 = resolved in complainant's favour)
outcome_prob_map = {
    "ATM / Debit Cards":              0.72,
    "Credit Cards":                   0.61,
    "Deposit Accounts":               0.68,
    "Electronic Banking / Mobile":    0.65,
    "Loans and Advances":             0.55,
    "Others":                         0.48,
    "Recovery Agents":                0.58,
    "Remittance / Money Transfer":    0.70,
}

# Generate resolution_days column
def gen_resolution_days(category):
    mean, std = resolution_time_map[category]
    days = int(np.random.normal(mean, std))
    return max(1, min(days, 60))  # clip between 1 and 60 days

# Generate outcome column
def gen_outcome(category):
    prob = outcome_prob_map[category]
    return int(np.random.random() < prob)

print("Generating synthetic labels...")
df['resolution_days'] = df['rbi_category'].apply(gen_resolution_days)
df['outcome']         = df['rbi_category'].apply(gen_outcome)

print(f"Done!\n")
print("Resolution days stats:")
print(df.groupby('rbi_category')['resolution_days'].mean().round(1))
print(f"\nOutcome distribution:")
print(df['outcome'].value_counts())
print(f"\nOutcome rate by category:")
print(df.groupby('rbi_category')['outcome'].mean().round(2))

Generating synthetic labels...
Done!

Resolution days stats:
rbi_category
ATM / Debit Cards               6.5
Credit Cards                   14.5
Deposit Accounts               11.5
Electronic Banking / Mobile     8.9
Loans and Advances             20.5
Others                         17.5
Recovery Agents                24.5
Remittance / Money Transfer     9.5
Name: resolution_days, dtype: float64

Outcome distribution:
outcome
1    1067936
0     955130
Name: count, dtype: int64

Outcome rate by category:
rbi_category
ATM / Debit Cards              0.73
Credit Cards                   0.61
Deposit Accounts               0.68
Electronic Banking / Mobile    0.44
Loans and Advances             0.55
Others                         0.48
Recovery Agents                0.58
Remittance / Money Transfer    0.70
Name: outcome, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score, classification_report
import numpy as np

# Save enriched dataset
df.to_csv('../data/processed/complaints_with_labels.csv', index=False)
print("Saved complaints_with_labels.csv\n")

# Sample 50k for training (same as Day 1)
df_sample = df.sample(n=50000, random_state=42).reset_index(drop=True)

# Features: just the label (category) + text length + amount mentioned
df_sample['text_length'] = df_sample['text'].str.len()
df_sample['has_amount']  = df_sample['text'].str.contains(
    r'\$[\d,]+|\d+\s*rupee|\d+\s*rs|\d+\s*lakh', 
    case=False, regex=True).astype(int)

features = ['label', 'text_length', 'has_amount']
X = df_sample[features]

# ── Regression: resolution days ───────────────────────────────
y_reg = df_sample['resolution_days']
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

reg_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train, y_train)
y_pred_reg = reg_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred_reg)
print(f"Resolution Time Predictor")
print(f"MAE: {mae:.2f} days  (target: ≤2 days)")
print(f"Sample predictions vs actual:")
for actual, pred in list(zip(y_test[:5], y_pred_reg[:5])):
    print(f"  Actual: {actual} days | Predicted: {pred:.1f} days")

# ── Binary classifier: outcome ────────────────────────────────
y_clf = df_sample['outcome']
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)

clf_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train2, y_train2)
y_pred_clf = clf_model.predict(X_test2)
acc = accuracy_score(y_test2, y_pred_clf)
print(f"\nOutcome Classifier")
print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test2, y_pred_clf,
    target_names=['Not in favour', 'In favour']))

Saved complaints_with_labels.csv

Resolution Time Predictor
MAE: 4.92 days  (target: ≤2 days)
Sample predictions vs actual:
  Actual: 36 days | Predicted: 24.5 days
  Actual: 14 days | Predicted: 17.6 days
  Actual: 20 days | Predicted: 17.6 days
  Actual: 26 days | Predicted: 17.5 days
  Actual: 16 days | Predicted: 20.6 days

Outcome Classifier
Accuracy: 0.5522
               precision    recall  f1-score   support

Not in favour       0.52      0.61      0.56      4705
    In favour       0.59      0.50      0.54      5295

     accuracy                           0.55     10000
    macro avg       0.56      0.56      0.55     10000
 weighted avg       0.56      0.55      0.55     10000



In [4]:
import joblib, os

# Save both models
os.makedirs('../models', exist_ok=True)
joblib.dump(reg_model, '../models/resolution_regressor.pkl')
joblib.dump(clf_model, '../models/outcome_classifier.pkl')
print("Models saved!")
print("  models/resolution_regressor.pkl")
print("  models/outcome_classifier.pkl")

# Install Whisper
import subprocess
result = subprocess.run(
    ['pip', 'install', 'openai-whisper'],
    capture_output=True, text=True
)
print("\nWhisper install output:")
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-300:] if result.stderr else "")
print("\nDone!")

Models saved!
  models/resolution_regressor.pkl
  models/outcome_classifier.pkl

Whisper install output:
--------- 3/5 [numba]
   ------------------------ --------------- 3/5 [numba]
   ------------------------ --------------- 3/5 [numba]
   ------------------------ --------------- 3/5 [numba]
   -------------------------------- ------- 4/5 [openai-whisper]
   -------------------------------- ------- 4/5 [openai-whisper]
   ---------------------------------------- 5/5 [openai-whisper]




Done!


In [5]:
import whisper
import torch

# Load Whisper small model (good balance of speed and accuracy)
# 'small' works well for Hindi + English on RTX 3050
print("Loading Whisper model...")
whisper_model = whisper.load_model("small")
print(f"Whisper loaded! Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Test with a synthetic audio file using gTTS
# Install gTTS to generate a test audio clip
import subprocess
subprocess.run(['pip', 'install', 'gtts', '-q'])

from gtts import gTTS
import os

# Create a test Hindi/English complaint audio
test_complaints = [
    ("en", "My ATM card was blocked and I cannot withdraw money from my account."),
    ("hi", "मेरे बैंक खाते से पैसे कट गए लेकिन ATM से पैसे नहीं निकले।"),
]

os.makedirs('../data/audio_samples', exist_ok=True)

for i, (lang, text) in enumerate(test_complaints):
    tts = gTTS(text=text, lang=lang)
    path = f'../data/audio_samples/test_{i}.mp3'
    tts.save(path)
    print(f"\nAudio {i+1} saved: {path}")
    print(f"Original: {text}")
    
    # Transcribe with Whisper
    result = whisper_model.transcribe(path)
    print(f"Transcribed: {result['text']}")
    print(f"Language detected: {result['language']}")

Loading Whisper model...


100%|███████████████████████████████████████| 461M/461M [01:30<00:00, 5.33MiB/s]


Whisper loaded! Device: GPU

Audio 1 saved: ../data/audio_samples/test_0.mp3
Original: My ATM card was blocked and I cannot withdraw money from my account.


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [7]:
import os

# Find the default folder where winget installs program links
winget_path = os.path.expanduser(r"~\AppData\Local\Microsoft\WinGet\Links")

# Inject it into the current notebook's memory
os.environ["PATH"] += os.pathsep + winget_path
print("Successfully updated PATH!")

Successfully updated PATH!


In [8]:
import whisper
import torch
from gtts import gTTS
import os

os.makedirs('../data/audio_samples', exist_ok=True)

# Generate English test audio
tts = gTTS(text="My ATM card was blocked and I cannot withdraw money.", lang='en')
tts.save('../data/audio_samples/test_en.mp3')
print("Audio saved!")

# Load Whisper and transcribe
print("Loading Whisper...")
whisper_model = whisper.load_model("small")
print("Transcribing...")
result = whisper_model.transcribe('../data/audio_samples/test_en.mp3')
print(f"\nTranscribed: {result['text']}")
print(f"Language: {result['language']}")

Audio saved!
Loading Whisper...
Transcribing...


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [1]:
import os
import sys
import whisper
import torch
from gtts import gTTS

# Fix ffmpeg PATH for Windows
ffmpeg_paths = [
    r"C:\Users\dixit\anaconda3\envs\nishkarsh\Library\bin",
    r"D:\conda-envs\nishkarsh\Library\bin",
]
for p in ffmpeg_paths:
    if os.path.exists(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + p
        print(f"Added to PATH: {p}")

# Verify ffmpeg is now findable
import subprocess
check = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if check.returncode == 0:
    print("ffmpeg found! ✅")
else:
    print("ffmpeg still not found ❌")

# Generate test audio
os.makedirs('../data/audio_samples', exist_ok=True)
tts = gTTS(text="My ATM card was blocked and I cannot withdraw money.", lang='en')
tts.save('../data/audio_samples/test_en.mp3')
print("Audio saved!")

# Load Whisper and transcribe
print("Loading Whisper...")
whisper_model = whisper.load_model("small")
result = whisper_model.transcribe('../data/audio_samples/test_en.mp3')
print(f"\nTranscribed: {result['text']}")
print(f"Language detected: {result['language']}")

ffmpeg found! ✅
Audio saved!
Loading Whisper...

Transcribed:  My ATM card was blocked and I cannot withdraw money.
Language detected: en


In [4]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import torch
import joblib
import numpy as np

# Load DistilBERT classifier
MODEL_PATH = '../models/distilbert-rbi-complaint'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_PATH)
bert_model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
bert_model.eval()

# Load regression + outcome models
reg_model = joblib.load('../models/resolution_regressor.pkl')
clf_model = joblib.load('../models/outcome_classifier.pkl')

LABELS = [
    "ATM / Debit Cards", "Credit Cards", "Deposit Accounts",
    "Electronic Banking / Mobile", "Loans and Advances",
    "Others", "Recovery Agents", "Remittance / Money Transfer"
]

def full_pipeline(text):
    """Takes raw complaint text → returns category, resolution days, outcome"""
    # Step 1: Classify category
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    label_idx = torch.argmax(probs).item()
    category = LABELS[label_idx]
    confidence = round(probs[label_idx].item() * 100, 2)

    # Step 2: Predict resolution time + outcome
    text_length = len(text)
    has_amount = int(bool(__import__('re').search(
        r'\$[\d,]+|\d+\s*rupee|\d+\s*rs|\d+\s*lakh', text, __import__('re').I)))
    features = [[label_idx, text_length, has_amount]]
    resolution_days = round(reg_model.predict(features)[0], 1)
    outcome = clf_model.predict(features)[0]
    outcome_text = "In complainant's favour" if outcome == 1 else "Not in favour"

    return {
        'category': category,
        'confidence': confidence,
        'resolution_days': resolution_days,
        'outcome': outcome_text
    }

def voice_pipeline(audio_path):
    """Takes audio file → transcribes → runs full pipeline"""
    print(f"Transcribing {audio_path}...")
    result = whisper_model.transcribe(audio_path)
    text = result['text'].strip()
    lang = result['language']
    print(f"Language: {lang} | Text: {text}")
    print()
    return full_pipeline(text)

# Test with 4 complaints — text + voice
print("=" * 55)
print("TEXT PIPELINE TESTS")
print("=" * 55)

test_complaints = [
    "My ATM card was blocked after wrong PIN. I need my money urgently.",
    "Bank deducted EMI twice for my home loan this month.",
    "I sent 50000 rupees via NEFT to wrong account number.",
    "Recovery agent is harassing me at home for loan repayment.",
]

for complaint in test_complaints:
    result = full_pipeline(complaint)
    print(f"Complaint: {complaint[:55]}...")
    print(f"Category:  {result['category']} ({result['confidence']}%)")
    print(f"Resolution: ~{result['resolution_days']} days")
    print(f"Outcome:   {result['outcome']}")
    print()

# Test voice pipeline
print("=" * 55)
print("VOICE PIPELINE TEST")
print("=" * 55)
result = voice_pipeline('../data/audio_samples/test_en.mp3')
print(f"Category:  {result['category']} ({result['confidence']}%)")
print(f"Resolution: ~{result['resolution_days']} days")
print(f"Outcome:   {result['outcome']}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

TEXT PIPELINE TESTS


D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names


Complaint: My ATM card was blocked after wrong PIN. I need my mone...
Category:  Deposit Accounts (91.47%)
Resolution: ~11.7 days
Outcome:   In complainant's favour

Complaint: Bank deducted EMI twice for my home loan this month....
Category:  Deposit Accounts (51.22%)
Resolution: ~12.0 days
Outcome:   In complainant's favour

Complaint: I sent 50000 rupees via NEFT to wrong account number....
Category:  Remittance / Money Transfer (95.31%)
Resolution: ~8.9 days
Outcome:   In complainant's favour

Complaint: Recovery agent is harassing me at home for loan repayme...
Category:  Loans and Advances (69.73%)
Resolution: ~20.7 days
Outcome:   Not in favour

VOICE PIPELINE TEST
Transcribing ../data/audio_samples/test_en.mp3...
Language: en | Text: My ATM card was blocked and I cannot withdraw money.

Category:  Deposit Accounts (93.69%)
Resolution: ~12.0 days
Outcome:   In complainant's favour


D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
D:\conda-envs\nishkarsh\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
